### Image Path Search

In [ ]:
import os
import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt
import re

In [ ]:
## 2024/07/03 search algorithm for image path

import os
import re
import pandas as pd

# Function to search for an image with a given pattern
def search_image(pattern, image_dir, show_path=False):
    for root, dirs, files in os.walk(image_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg')):
                # Extract filename without extension
                filename_without_ext = os.path.splitext(file)[0]
                # Check for exact match
                if re.fullmatch(pattern, filename_without_ext, re.IGNORECASE):
                    if show_path:
                        return os.path.join(root, file)
                    else:
                        return filename_without_ext
    return None

# Function to find image file name with the given item number
def find_image_file_name(item_number):
    image_dir = os.path.join(os.getcwd(), 'data/images')
    image_map_file = os.path.join(image_dir, 'image_map.csv')

    # Initialize result variables
    r1_exact = None
    r2_character = None
    r3_parts = None
    r4_map = None

    # Rule 1: Search for exact match with full item number
    r1_exact = search_image(re.escape(item_number), image_dir)
    
    # Rule 2: Search after last character removed
    if item_number[-1] in "ABCDSMLFE":
        item_number_modified = item_number[:-1]
        r2_character = search_image(re.escape(item_number_modified), image_dir)
    
    # Rule 3: Partial search without the last '-'
    parts = item_number.split('-')
    if len(parts) >= 2:
        partial_pattern = re.escape(parts[0] + '-' + parts[1])
        r3_parts = search_image(partial_pattern, image_dir)
    
    # Rule 4: Use image log
    if os.path.exists(image_map_file):
        image_map_df = pd.read_csv(image_map_file)
        # Search for the item number in the image map
        image_file_name = image_map_df.loc[image_map_df['item_number'] == item_number, 'image_file_name']
        if not image_file_name.empty:
            # Construct the full path to the image file with .jpg extension
            image_map_name = image_file_name.values[0]
            if isinstance(image_map_name, str):
                r4_map = search_image(re.escape(image_map_name), image_dir) # Escape the string to use as a regex pattern

    # Return the first non-None result
    if r1_exact:
        return r1_exact
    elif r2_character:
        return r2_character
    elif r3_parts:
        return r3_parts
    elif r4_map:
        return r4_map
    else:
        return None # If no match found, return None

# find_image_file_name('CS-022-100GD')
find_image_file_name('CS-GB60')
    

### Seasonal Trends

In [ ]:
## In customer analysis

# Toggle between 'Seasonal Trends in Sales (Quantity)' and 'Seasonal Trends in Sales (Revenue)'
col1, col2 = st.columns([2.5, 7.5])
col1.write("### Seasonal Trends")
# check_seasonal_trends = col2.checkbox("By Revenue", key='seasonal_trend_revenue')

filtered_data['month'] = pd.to_datetime(filtered_data['date']).dt.month
seasonal_trends = filtered_data.groupby('month').agg({'quantity': 'sum', 'retail': 'sum'}).reset_index()

if metric_option == 'Revenue':
    fig_seasonal_trends_revenue = px.line(seasonal_trends, x='month', y='retail', title='Seasonal Trends in Sales (Revenue)', markers=True)
    fig_seasonal_trends_revenue.update_layout(xaxis_title='Month', yaxis_title='Revenue')
    st.plotly_chart(fig_seasonal_trends_revenue)
else:
    fig_seasonal_trends_quantity = px.line(seasonal_trends, x='month', y='quantity', title='Seasonal Trends in Sales (Quantity)', markers=True)
    fig_seasonal_trends_quantity.update_layout(xaxis_title='Month', yaxis_title='Quantity Sold')
    st.plotly_chart(fig_seasonal_trends_quantity)

In [ ]:
## 2024/07/06: removed sales timeline from sales_analysis.py

# Display the ranked customers
st.write("### Sales Timeline")
# st.data_editor(customer_rank, use_container_width=True)

# Aggregate the quantity of sales over time
sales_over_time = filtered_data.groupby('date')['quantity'].sum().reset_index()

# Create a DataFrame with accno and name for hover info
hover_info = filtered_data.groupby('date').agg({
    'accno': 'first', 'name': 'first', 'quantity': 'sum'
}).reset_index()

# Plot the sales over time using Plotly with accno and name annotations
fig = px.line(sales_over_time, x='date', y='quantity', markers=True)
fig.update_layout(xaxis_title='Date', yaxis_title='Quantity Sold', height=300)  # Adjust height here

# Add customer codes (accno) above each point
for idx, row in sales_over_time.iterrows():
    accno = filtered_data[filtered_data['date'] == row['date']]['accno'].iloc[0]
    fig.add_annotation(x=row['date'], y=row['quantity'], text=accno, showarrow=True, arrowhead=2, ax=0, ay=-20)

# Add hover information with accno and name
fig.update_traces(
    hovertemplate="<b>ACCNO</b>: %{customdata[0]}<br><b>Name</b>: %{customdata[1]}<br><b>Quantity</b>: %{y}<br><b>Date</b>: %{x}",
    customdata=hover_info[['accno', 'name']].values
)

st.plotly_chart(fig)

## removed from sales_overview.py, under 'Graph option':

In [ ]:
## Item Sales Rank

    elif display_option == 'Graph':
        with col1:
            selected_points = item_rank_graph(df=ranked_sales, metric_option=metric_option)
        with col2:
            if selected_points:
                point_index = selected_points[0]['pointIndex']
                item_number = ranked_sales['item_number'].iloc[point_index]
                show_item_image(items_df, item_number)

                # Calculate metrics
                item_filtered_data = filtered_data[filtered_data['item_number'] == item_number]
                total_customers = item_filtered_data['accno'].nunique()
                total_purchases = item_filtered_data.shape[0]
                total_quantity_sold = item_filtered_data['quantity'].sum()
                total_revenue = item_filtered_data['retail'].sum()
                current_stock = items_df[items_df['item_number'] == item_number]['instock'].iloc[0]
                purchase_dates = item_filtered_data['date'].sort_values().unique()
                if len(purchase_dates) > 1:
                    purchase_intervals = pd.Series(purchase_dates).diff().dropna().dt.days
                    average_purchase_frequency = purchase_intervals.mean()
                else:
                    average_purchase_frequency = 'N/A'

                col1, col2 = st.columns(2)
                if metric_option == 'retail':
                    col1.metric('Total Revenue', total_revenue)
                else:
                    col1.metric('Total Quantity Sold', total_quantity_sold)
                col2.metric('Current Stock', int(current_stock))

                if st.button("See Details"):
                    st.session_state.current_page = "Sales Analysis"
                    st.session_state.selected_item_number = item_number
                    st.experimental_rerun()
            else:
                item_number = ranked_sales['item_number'].iloc[0]
                show_item_image(items_df, item_number)

                # Calculate metrics
                item_filtered_data = filtered_data[filtered_data['item_number'] == item_number]
                total_customers = item_filtered_data['accno'].nunique()
                total_purchases = item_filtered_data.shape[0]
                total_quantity_sold = item_filtered_data['quantity'].sum()
                current_stock = items_df[items_df['item_number'] == item_number]['instock'].iloc[0]
                purchase_dates = item_filtered_data['date'].sort_values().unique()
                if len(purchase_dates) > 1:
                    purchase_intervals = pd.Series(purchase_dates).diff().dropna().dt.days
                    average_purchase_frequency = purchase_intervals.mean()
                else:
                    average_purchase_frequency = 'N/A'

                col1, col2 = st.columns(2)
                if metric_option == 'retail':
                    col1.metric('Total Revenue', total_revenue)
                else:
                    col1.metric('Total Quantity Sold', total_quantity_sold)
                col2.metric('Current Stock', int(current_stock))

                if st.button("See Details", key='item_analysis_details'):
                    st.session_state.current_page = "Sales Analysis"
                    st.session_state.selected_item_number = item_number
                    st.experimental_rerun()

In [ ]:
## From Customer Ranking
    else:
        with col1:
            selected_points = customer_ranking_graph(df=customer_ranking, metric_option=metric_option, graph_height=None)
        with col2:
            if selected_points:
                point_index = selected_points[0]['pointIndex']
                selected_customer_number = customer_ranking['accno'].iloc[point_index]
            else:
                selected_customer_number = customer_ranking['accno'].iloc[0]
            customer_preview(accno=selected_customer_number, accounts_df=accounts_df)
    with col2:
        if st.button("See Details", key='customer_analysis_details'):
                st.session_state.current_page = "Customer Analysis"
                st.session_state.selected_customer_number = selected_customer_number
                st.experimental_rerun()

## sales_analysis

In [ ]:
## metrics for purchase interval
    # Average day between 2 purchases
    purchase_dates = filtered_data['date'].sort_values().unique()
    if len(purchase_dates) > 1:
        purchase_intervals = pd.Series(purchase_dates).diff().dropna().dt.days
        average_purchase_frequency = int(round(purchase_intervals.mean(), 0))

In [ ]:
## might need: changing colour of the item in the same series table
    def color_red_row(row):
        index_to_highlight = 2  # specify the index of the row to highlight
        return ['color: blue' if row.name == index_to_highlight else '' for _ in row]
    same_series_item = same_series_item.style.apply(color_red_row, axis=1)

## ADD BACK v.3: Sales Overview

In [ ]:


    # Third graph: Sales Over Time
    st.subheader('Sales Over Time')
    col1, col2 = st.columns([6.5, 3.5])

    revenue_over_time = revenue_over_time[['date', 'retail', 'quantity']].sort_values(by='date', ascending=False).reset_index(drop=True)
    filtered_data['date'] = pd.to_datetime(filtered_data['date'])
    with col1:
    #     on = st.toggle(label='sales_over_time_table')
    #     if on:
    #         # st.dataframe(revenue_over_time[['date', 'retail', 'quantity']].sort_values(by='date', ascending=False).reset_index(drop=True), use_container_width=True, height=500)
        

    #         selected_item_row = st.dataframe(revenue_over_time,
    #                                     hide_index=True, on_select='rerun', selection_mode="single-row")

    #         # click to check transaction of the day
    #         selected_rowno = selected_item_row.selection.rows
    #         item_number_colno = revenue_over_time.columns.get_loc('date')
    #         selected_df = revenue_over_time.iloc[selected_rowno, item_number_colno]
    #         if not selected_df.empty:
    #             date = selected_df.iloc[0]
    #             selected_date = date
    #         else:
    #             selected_date = pd.to_datetime(revenue_over_time['date'][0])
            
        
    #     else:
        selected_points = sales_over_time_graph(df=revenue_over_time, metric_option=metric_option)
        
        if selected_points:
            point_index = selected_points[0]['pointIndex']
            selected_date = pd.to_datetime(revenue_over_time['date'][point_index])
        else:
            selected_date = pd.to_datetime(revenue_over_time['date'][0])

    with col2:
        daily_transaction(df=filtered_data, date=selected_date)

In [ ]:
## v.3: plotly event for graphs
selected_points = plotly_events(fig3, click_event=True)

## stock change calculation with print

In [ ]:


def plot_stock_history(sales_data=None,
                        purchase_record_df=None,
                        accounts_df=None,
                        item_number=None,
                        items_df=None,
                        date_start=None,
                        date_end=None,
                        xrange=None,
                        yrange=None,
                        height=None,
                        show_accno=True,
                        title=None):
    # Get the current stock count and its date
    # current_stock_row = stock_count_df[stock_count_df['item_number'] == item_number]
    # if current_stock_row.empty:
    #     current_stock_row = pd.DataFrame({'item_number': item_number, 'quantity': 0, 'date': dt.datetime.today()}, index=[0])

    current_stock = items_df[items_df['item_number'] == item_number]['instock'].iloc[0]
    st.write('current_stock')
    st.write(current_stock)
    # current_stock_date = pd.to_datetime(current_stock_row['date'].values[0])

    # # Get the sales transactions for the item
    # sales_transactions = sales_data[sales_data['item_number'] == item_number][['date', 'quantity']]

    # Get the sales transactions for the item
    sales_transactions = sales_data[(sales_data['date'] >= date_start) & 
                                        (sales_data['date'] <= date_end) &
                                        (sales_data['item_number'] == item_number)][['date', 'quantity', 'accno']]

    st.write('sales transactions')
    st.write(sales_transactions)

    # Get the purchase records for the item within the date range
    purchase_records = purchase_record_df[(purchase_record_df['date'] >= date_start) & 
                                        (purchase_record_df['date'] <= date_end) &
                                        (purchase_record_df['item_number'] == item_number)][['date', 'quantity']]

    purchase_records['accno'] = 'STOCK-IN'

    st.write('purchase_records')
    st.write(purchase_records)

    # Combine sales and purchase data
    sales_transactions['quantity'] = -sales_transactions['quantity']  # Sales reduce stock
    purchase_records['quantity'] = purchase_records['quantity']      # Purchases increase stock

    st.write('''sales_transactions['quantity'] = -sales_transactions['quantity']''')
    st.write('sales_transactions')
    st.write(sales_transactions)

    st.write('''purchase_records['quantity'] = purchase_records['quantity']''')
    st.write('purchase_records')
    st.write(purchase_records)

    stock_changes = pd.concat([sales_transactions, purchase_records])

    st.write('stock_changes (conbined)')
    st.write(stock_changes)

    # Sort by date descending to count backwards
    stock_changes = stock_changes.sort_values(by='date', ascending=False)
    st.write('stock_changes sorted')
    st.write(stock_changes)

    # Calculate cumulative stock backwards
    stock_changes['cumulative_quantity'] = stock_changes['quantity'].cumsum()
    st.write('''stock_changes['cumulative_quantity'] = stock_changes['quantity'].cumsum()''')
    st.write('stock_changes')
    st.write(stock_changes)

    # The cumulative stock starting from the current stock gives us the historical stock levels
    stock_changes['stock_count'] = current_stock - stock_changes['cumulative_quantity']
    st.write('''stock_changes['stock_count'] = current_stock - stock_changes['cumulative_quantity']''')
    st.write('stock_changes')
    st.write(stock_changes)

    # Correct the stock count by shifting back one date point
    stock_changes['stock_count'] = stock_changes['stock_count'].shift(1)
    stock_changes = stock_changes.fillna(current_stock)
    st.write('''stock_changes['stock_count'] = stock_changes['stock_count'].shift(1)''')
    st.write('''stock_changes = stock_changes.fillna(current_stock)''')
    st.write('stock_changes')
    st.write(stock_changes)

    # Sort by date ascending for plotting
    stock_changes = stock_changes.sort_values(by='date', ascending=True).reset_index(drop=True)
    st.write('stock_changes sorted by date')
    st.write(stock_changes)

    # Add customer names for hover info
    stock_changes = pd.merge(stock_changes, accounts_df[['accno', 'name']], on='accno', how='left')
    st.write('stock_changes merged with accno name')
    st.write(stock_changes)


    # Add stock in our out text
    def pos_sign(num):
        if num>0: return '+' + str(num)
        else: return str(num)
    stock_changes['stock_inout'] = stock_changes['quantity'].apply(pos_sign)
    st.write('add stock_inout column')
    st.write(stock_changes)



    # Plot the stock count history with markers
    # fig = px.line(stock_changes, x='Date', y='Stock Count', title=f'Stock Count History for {item_number}', markers=True)
    # fig.update_traces(mode='lines+markers')
    fig = px.line(stock_changes, x='date', y='stock_count', markers=True, title=title)
    fig.update_layout(xaxis_title='Date', yaxis_title='Stock Count', height=height, margin=dict(t=0, b=0), hoverlabel=dict(bgcolor="white", font_color='black', font_size=14))  # Adjust height here



    # Add customer codes (accno) above each point
    if show_accno == True:
        for idx, row in stock_changes.iterrows():
            accno = stock_changes[stock_changes['date'] == row['date']]['accno'].iloc[0]
            fig.add_annotation(x=row['date'], y=row['stock_count'], text=accno, showarrow=True, arrowhead=2, ax=0, ay=-20)

    # Add hover information with accno and name
    fig.update_traces(
        hovertemplate="<b>Account No.</b>: %{customdata[0]}<br><b>Name</b>: %{customdata[1]}<br><b>In/Out</b>: %{customdata[2]}<br><b>In Stock</b>: %{y}<br><b>Date</b>: %{x}",
        customdata=stock_changes[['accno', 'name', 'stock_inout']].values
    )

    fig.update_xaxes(range=xrange)
    fig.update_yaxes(range=yrange)

    st.plotly_chart(fig, use_container_width=True)

    return stock_changes

## removed find_image_path from utils


In [ ]:

# Function to find image path with the given item number
def find_image_path(item_number):
    image_dir = os.path.join(os.getcwd(), 'data/images')
    image_map_file = os.path.join(image_dir, 'image_map.csv')

    # Function to search for an image with a given pattern
    def search_image(pattern):
        for root, dirs, files in os.walk(image_dir):
            for file in files:
                if file.lower().endswith(('.jpg', '.jpeg', 'png')):
                    # Extract filename without extension
                    filename_without_ext = os.path.splitext(file)[0]
                    # Check for exact match
                    if re.fullmatch(pattern, filename_without_ext, re.IGNORECASE):
                        return os.path.join(root, file)
        return None

    # Search for exact match with full item number
    image_path = search_image(re.escape(item_number))
    if image_path:
        return image_path

    # If full item number match not found, perform partial search
    parts = item_number.split('-')
    if len(parts) >= 2:
        partial_pattern = re.escape(parts[0] + '-' + parts[1])
        image_path = search_image(partial_pattern)
        if image_path:
            return image_path

    # If still not found, check if the last character is A, B, C, D, S.... etc. and remove it
    if item_number[-1] in "ABCDSMLFE":
        item_number_modified = item_number[:-1]
        image_path = search_image(re.escape(item_number_modified))
        if image_path:
            return image_path

        # Perform partial search with modified item number
        parts_modified = item_number_modified.split('-')
        if len(parts_modified) >= 2:
            partial_pattern_modified = re.escape(parts_modified[0] + '-' + parts_modified[1])
            image_path = search_image(partial_pattern_modified)
            if image_path:
                return image_path

    # Load the image map from the CSV file
    if os.path.exists(image_map_file):
        image_map_df = pd.read_csv(image_map_file)
        # Search for the item number in the image map
        image_file_name = image_map_df.loc[image_map_df['item_number'] == item_number, 'image_file_name']
        # NOTES FOR NAMING PURPOSES
        # notes = image_map_df.loc[image_map_df['item_number'] == item_number, 'notes']
        if not image_file_name.empty:
            # Construct the full path to the image file with .jpg extension
            image_map_name = image_file_name.values[0]
            if isinstance(image_map_name, str):
                image_map_name = re.escape(image_map_name)  # Escape the string to use as a regex pattern
                image_path = search_image(image_map_name)
                # st.write(f"{item_number} ({notes.values[0]})")  # NAMING NOTES
                if image_path:
                    return image_path
            else:
                st.write(f"Error: image_map_name is not a string. Value: {image_map_name}")

    # If no match found, return None
    return None